In [ ]:
# CPU allocation: set an inclusive range, or leave both as None to retain scheduler affinity.
import os

CPU_START = None
CPU_STOP = None
if CPU_START is not None and CPU_STOP is not None:
    cpus = set(range(int(CPU_START), int(CPU_STOP) + 1))
    os.sched_setaffinity(0, cpus)
    os.environ["OMP_NUM_THREADS"] = str(len(cpus))
    os.environ["OPENBLAS_NUM_THREADS"] = str(len(cpus))
    os.environ["MKL_NUM_THREADS"] = str(len(cpus))
print("Notebook CPU affinity:", sorted(os.sched_getaffinity(0))[:8], "...", sorted(os.sched_getaffinity(0))[-8:], "count", len(os.sched_getaffinity(0)))


# N16 Fine-Grid Charge-Sharpening Analysis (`dw_truncation=True`)

This notebook analyzes the completed CPU campaign
`N16_alpha-fine21_nsh1_dwtrunc1_init-maxmix_S10_cycles-2Ny` without rerunning dynamics.
The observables are trajectory-resolved global entropy and total charge variance collected once per completed cycle from a maximally mixed initial state.


In [ ]:
from __future__ import annotations

import json
import math
import os
import sys
from pathlib import Path

import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / 'colab_charge_fluctuations').exists():
    for parent in Path.cwd().parents:
        if (parent / 'colab_charge_fluctuations').exists():
            REPO_ROOT = parent
            break

BUNDLE_ROOT = REPO_ROOT / 'colab_charge_fluctuations'
SRC_ROOT = BUNDLE_ROOT / 'src'
if str(SRC_ROOT) not in sys.path:
    sys.path.insert(0, str(SRC_ROOT))

from charge_sharpening_alpha_sweep_loader import SHARPENING_THRESHOLD, load_charge_sharpening_campaign

CAMPAIGN_ID = 'N16_alpha-fine21_nsh1_dwtrunc1_init-maxmix_S10_cycles-2Ny'
OUTPUT_ROOT = BUNDLE_ROOT / 'analysis_outputs' / 'purification_charge_sharpening_alpha_sweep_cpu' / CAMPAIGN_ID
FIGURE_ROOT = OUTPUT_ROOT / 'figures'
TABLE_ROOT = OUTPUT_ROOT / 'tables'
FIGURE_ROOT.mkdir(parents=True, exist_ok=True)
TABLE_ROOT.mkdir(parents=True, exist_ok=True)

payload = load_charge_sharpening_campaign(BUNDLE_ROOT, campaign_id=CAMPAIGN_ID, require_complete=True)
inventory_df = payload['inventory_df']
trajectory_df = payload['trajectory_df']
steady_state_df = payload['steady_state_df']
critical_diagnostics_df = payload['critical_diagnostics_df']

alpha_grid = sorted(inventory_df['alpha_topological_region'].unique())
ny_values = sorted(inventory_df['Ny'].unique())
protocols = ['perfect_correction', 'postselection']

# Domain-wall slab width for Nx=16: DW_loc=[Nx//6, 5Nx//6] inclusive.
def measured_modes(nx: int, ny: int) -> int:
    slab_width = 2 * nx // 3 + 1
    return 2 * slab_width * ny

steady_state_df = steady_state_df.copy()
steady_state_df['measured_modes'] = steady_state_df.apply(lambda r: measured_modes(int(r['Nx']), int(r['Ny'])), axis=1)
steady_state_df['entropy_density_mean'] = steady_state_df['total_entropy_mean'] / steady_state_df['measured_modes']
steady_state_df['entropy_density_std'] = steady_state_df['total_entropy_std'] / steady_state_df['measured_modes']
steady_state_df['charge_variance_density_mean'] = steady_state_df['total_charge_variance_mean'] / steady_state_df['measured_modes']
steady_state_df['charge_variance_density_std'] = steady_state_df['total_charge_variance_std'] / steady_state_df['measured_modes']

trajectory_df = trajectory_df.copy()
trajectory_df['measured_modes'] = trajectory_df.apply(lambda r: measured_modes(int(r['Nx']), int(r['Ny'])), axis=1)
trajectory_df['entropy_density'] = trajectory_df['total_entropy'] / trajectory_df['measured_modes']
trajectory_df['charge_variance_density'] = trajectory_df['total_charge_variance'] / trajectory_df['measured_modes']

summary = {
    'campaign_id': CAMPAIGN_ID,
    'inventory_rows': int(len(inventory_df)),
    'trajectory_rows': int(len(trajectory_df)),
    'steady_state_rows': int(len(steady_state_df)),
    'protocols': protocols,
    'Ny_values': [int(v) for v in ny_values],
    'alpha_grid': [float(v) for v in alpha_grid],
    'dw_truncation_values': sorted(bool(v) for v in inventory_df['dw_truncation'].unique()),
    'observable_regions': sorted(inventory_df['observable_region'].unique()),
    'sharpening_threshold': SHARPENING_THRESHOLD,
}

with (TABLE_ROOT / 'analysis_summary.json').open('w', encoding='utf-8') as fh:
    json.dump(summary, fh, indent=2)
inventory_df.to_csv(TABLE_ROOT / 'inventory.csv', index=False)
trajectory_df.to_parquet(TABLE_ROOT / 'trajectory_observables.parquet', index=False)
steady_state_df.to_csv(TABLE_ROOT / 'steady_state_summary.csv', index=False)
critical_diagnostics_df.to_csv(TABLE_ROOT / 'critical_projector_diagnostics.csv', index=False)

print(json.dumps(summary, indent=2))


## Completeness And Metadata


In [ ]:
expected_inventory = 2 * 3 * 21
expected_steady = expected_inventory
expected_trajectory = 21 * (10 + 1) * (2 * 16 + 2 * 24 + 2 * 32)

assert len(inventory_df) == expected_inventory, len(inventory_df)
assert len(steady_state_df) == expected_steady, len(steady_state_df)
assert len(trajectory_df) == expected_trajectory, len(trajectory_df)
assert inventory_df['complete'].all()
assert sorted(inventory_df['Ny'].unique()) == [16, 24, 32]
assert len(alpha_grid) == 21 and np.allclose(alpha_grid, np.linspace(1.0, 3.0, 21))
assert set(inventory_df['dw_truncation'].unique()) == {True}
assert set(inventory_df['observable_region'].unique()) == {'domain_wall_slab'}
assert np.isfinite(trajectory_df[['total_entropy', 'total_charge_variance']].to_numpy()).all()

counts = inventory_df.groupby(['protocol'], as_index=False).agg(
    configs=('config_id', 'count'),
    samples_completed=('samples_completed', 'sum'),
    min_cycles=('cycles', 'min'),
    max_cycles=('cycles', 'max'),
)
display(counts)
display(inventory_df.groupby(['protocol', 'Ny'], as_index=False).agg(configs=('config_id','count'), samples=('samples_completed','sum')))
display(critical_diagnostics_df)


## Figure Style And Derived Tables


In [ ]:
try:
    mpl.rcParams['font.family'] = 'CMU Sans Serif'
except Exception:
    mpl.rcParams['font.family'] = 'DejaVu Sans'
mpl.rcParams.update({
    'figure.dpi': 150,
    'savefig.dpi': 300,
    'font.size': 8,
    'axes.titlesize': 9,
    'axes.labelsize': 9,
    'xtick.labelsize': 8,
    'ytick.labelsize': 8,
    'legend.fontsize': 7,
    'mathtext.fontset': 'cm',
    'axes.spines.top': False,
    'axes.spines.right': False,
})

ny_colors = {16: '#0072B2', 24: '#D55E00', 32: '#009E73'}
protocol_styles = {
    'perfect_correction': {'linestyle': '-', 'label': 'perfect correction'},
    'postselection': {'linestyle': '--', 'label': 'post-selection'},
}
alpha_ticks = [1.0, 1.5, 2.0, 2.5, 3.0]

final_df = trajectory_df.loc[
    trajectory_df['cycle'] == trajectory_df.groupby('config_id')['cycle'].transform('max')
].copy()
final_df.to_csv(TABLE_ROOT / 'final_cycle_trajectory_values.csv', index=False)

# Summary including quantiles to make the crossover/sample spread explicit.
final_summary = final_df.groupby(['protocol', 'Ny', 'alpha_topological_region'], as_index=False).agg(
    samples=('sample_index', 'count'),
    entropy_mean=('total_entropy', 'mean'),
    entropy_std=('total_entropy', lambda s: s.std(ddof=1) if len(s) > 1 else 0.0),
    entropy_median=('total_entropy', 'median'),
    entropy_q10=('total_entropy', lambda s: s.quantile(0.10)),
    entropy_q90=('total_entropy', lambda s: s.quantile(0.90)),
    charge_variance_mean=('total_charge_variance', 'mean'),
    charge_variance_std=('total_charge_variance', lambda s: s.std(ddof=1) if len(s) > 1 else 0.0),
    charge_variance_median=('total_charge_variance', 'median'),
    charge_variance_q10=('total_charge_variance', lambda s: s.quantile(0.10)),
    charge_variance_q90=('total_charge_variance', lambda s: s.quantile(0.90)),
    sharpening_fraction=('total_charge_variance', lambda s: float(np.mean(s < SHARPENING_THRESHOLD))),
    measured_modes=('measured_modes', 'first'),
).sort_values(['protocol', 'Ny', 'alpha_topological_region'])
final_summary['entropy_density_mean'] = final_summary['entropy_mean'] / final_summary['measured_modes']
final_summary['entropy_density_std'] = final_summary['entropy_std'] / final_summary['measured_modes']
final_summary['charge_variance_density_mean'] = final_summary['charge_variance_mean'] / final_summary['measured_modes']
final_summary['charge_variance_density_std'] = final_summary['charge_variance_std'] / final_summary['measured_modes']
final_summary.to_csv(TABLE_ROOT / 'final_cycle_summary_with_quantiles.csv', index=False)

dynamics_df = trajectory_df.groupby(['protocol', 'Ny', 'alpha_topological_region', 'cycle'], as_index=False).agg(
    entropy_mean=('total_entropy', 'mean'),
    entropy_std=('total_entropy', lambda s: s.std(ddof=1) if len(s) > 1 else 0.0),
    charge_variance_mean=('total_charge_variance', 'mean'),
    charge_variance_std=('total_charge_variance', lambda s: s.std(ddof=1) if len(s) > 1 else 0.0),
)
dynamics_df.to_csv(TABLE_ROOT / 'cycle_dynamics_summary.csv', index=False)

def savefig(fig, name):
    png = FIGURE_ROOT / f'{name}.png'
    pdf = FIGURE_ROOT / f'{name}.pdf'
    fig.savefig(png, bbox_inches='tight', dpi=300)
    fig.savefig(pdf, bbox_inches='tight')
    print(png)
    return png


## Final-Cycle Raw Observables Versus $lpha_1$


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(6.75, 4.85), sharex=True, constrained_layout=True)
observables = [
    ('entropy_mean', 'entropy_std', r'$S_{\mathrm{global}}(2N_y)$', 'Global entropy'),
    ('charge_variance_mean', 'charge_variance_std', r'$\delta Q^2(2N_y)$', 'Total charge variance'),
]
for row, protocol in enumerate(protocols):
    for col, (mean_col, std_col, ylabel, obs_title) in enumerate(observables):
        ax = axes[row, col]
        for ny in ny_values:
            sub = final_summary[(final_summary['protocol'] == protocol) & (final_summary['Ny'] == ny)]
            x = sub['alpha_topological_region'].to_numpy(float)
            y = sub[mean_col].to_numpy(float)
            yerr = sub[std_col].to_numpy(float)
            ax.plot(x, y, marker='o', ms=2.8, lw=1.35, color=ny_colors[ny], linestyle='-',
                    label=fr'$N_y={ny}$')
            if protocol == 'perfect_correction':
                ax.fill_between(x, y - yerr, y + yerr, color=ny_colors[ny], alpha=0.14, lw=0,
                                label=fr'$N_y={ny}$ $\pm 1\sigma$' if col == 0 else None)
        ax.axvline(2.0, color='0.35', linestyle=':', lw=1.1)
        ax.set_title(f'{protocol_styles[protocol]["label"]}: {obs_title}')
        ax.set_xlabel(r'$\alpha_1$')
        ax.set_ylabel(ylabel)
        ax.set_xticks(alpha_ticks)
        ax.legend(loc='best', frameon=False, title='System size')
savefig(fig, 'fine21_final_raw_observables_vs_alpha')
plt.show()


## Final-Cycle Observable Densities Versus $lpha_1$


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(6.75, 4.85), sharex=True, constrained_layout=True)
observables = [
    ('entropy_density_mean', 'entropy_density_std', r'$S_{\mathrm{global}}/N_{\mathrm{meas}}$', 'Entropy density'),
    ('charge_variance_density_mean', 'charge_variance_density_std', r'$\delta Q^2/N_{\mathrm{meas}}$', 'Charge-variance density'),
]
for row, protocol in enumerate(protocols):
    for col, (mean_col, std_col, ylabel, obs_title) in enumerate(observables):
        ax = axes[row, col]
        for ny in ny_values:
            sub = final_summary[(final_summary['protocol'] == protocol) & (final_summary['Ny'] == ny)]
            x = sub['alpha_topological_region'].to_numpy(float)
            y = sub[mean_col].to_numpy(float)
            yerr = sub[std_col].to_numpy(float)
            ax.plot(x, y, marker='o', ms=2.8, lw=1.35, color=ny_colors[ny], linestyle='-',
                    label=fr'$N_y={ny}$')
            if protocol == 'perfect_correction':
                ax.fill_between(x, y - yerr, y + yerr, color=ny_colors[ny], alpha=0.14, lw=0,
                                label=fr'$N_y={ny}$ $\pm 1\sigma$' if col == 0 else None)
        ax.axvline(2.0, color='0.35', linestyle=':', lw=1.1)
        ax.set_title(f'{protocol_styles[protocol]["label"]}: {obs_title}')
        ax.set_xlabel(r'$\alpha_1$')
        ax.set_ylabel(ylabel)
        ax.set_xticks(alpha_ticks)
        ax.legend(loc='best', frameon=False, title='System size')
savefig(fig, 'fine21_final_density_observables_vs_alpha')
plt.show()


## Final-Cycle Sharpening Fraction


In [ ]:
fig, ax = plt.subplots(figsize=(3.375, 2.6), constrained_layout=True)
for protocol in protocols:
    for ny in ny_values:
        sub = final_summary[(final_summary['protocol'] == protocol) & (final_summary['Ny'] == ny)]
        style = protocol_styles[protocol]
        ax.plot(sub['alpha_topological_region'], sub['sharpening_fraction'], marker='o', ms=3.0, lw=1.35,
                color=ny_colors[ny], linestyle=style['linestyle'], label=fr'$N_y={ny}$, {style["label"]}')
ax.axvline(2.0, color='0.35', linestyle=':', lw=1.1)
ax.set_xlabel(r'$\alpha_1$')
ax.set_ylabel(fr'$\Pr(\delta Q^2 < {SHARPENING_THRESHOLD:g})$')
ax.set_ylim(-0.05, 1.05)
ax.set_xticks(alpha_ticks)
ax.legend(ncol=1, loc='center left', bbox_to_anchor=(1.02, 0.5), frameon=False)
savefig(fig, 'fine21_final_sharpening_fraction_vs_alpha')
plt.show()


## Crossover Heatmaps


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(6.75, 4.7), constrained_layout=True, sharex=True, sharey=True)
plot_specs = [
    ('perfect_correction', 'entropy_mean', r'$S_{\mathrm{global}}$'),
    ('postselection', 'entropy_mean', r'$S_{\mathrm{global}}$'),
    ('perfect_correction', 'charge_variance_mean', r'$\delta Q^2$'),
    ('postselection', 'charge_variance_mean', r'$\delta Q^2$'),
]
for ax, (protocol, value_col, label) in zip(axes.flat, plot_specs):
    pivot = final_summary[final_summary['protocol'] == protocol].pivot(index='Ny', columns='alpha_topological_region', values=value_col)
    data = pivot.loc[ny_values, alpha_grid].to_numpy(float)
    im = ax.imshow(data, aspect='auto', origin='lower', interpolation='nearest',
                   extent=[min(alpha_grid), max(alpha_grid), min(ny_values), max(ny_values)], cmap='viridis')
    ax.axvline(2.0, color='w', linestyle=':', lw=1.0)
    ax.set_title(f'{protocol.replace("_", " ")}: {label}')
    ax.set_xlabel(r'$\alpha_1$')
    ax.set_ylabel(r'$N_y$')
    ax.set_yticks(ny_values)
    fig.colorbar(im, ax=ax, shrink=0.85)
savefig(fig, 'fine21_final_observable_heatmaps')
plt.show()


## Perfect-Correction Final-Cycle Spread Around The Crossover


In [ ]:
spread_alphas = [1.3, 1.4, 1.5, 1.6, 1.7, 1.8, 1.9, 2.0, 2.1]
spread_df = final_df[(final_df['protocol'] == 'perfect_correction') & (final_df['alpha_topological_region'].round(10).isin(spread_alphas))].copy()
mean_spread = spread_df.groupby(['Ny', 'alpha_topological_region'], as_index=False).agg(
    total_entropy=('total_entropy', 'mean'),
    total_charge_variance=('total_charge_variance', 'mean'),
)

fig, axes = plt.subplots(1, 2, figsize=(6.75, 2.75), constrained_layout=True)
for ax, value_col, ylabel, floor, title in [
    (axes[0], 'total_entropy', r'$S_{\mathrm{global}}$', 1e-8, 'Entropy spread'),
    (axes[1], 'total_charge_variance', r'$\delta Q^2$', 1e-12, 'Charge-variance spread'),
]:
    for ny in ny_values:
        sub = spread_df[spread_df['Ny'] == ny]
        x = sub['alpha_topological_region'].to_numpy(float)
        jitter = (sub['sample_index'].to_numpy(float) - sub['sample_index'].mean()) * 0.0025
        marker = 'o' if ny == 16 else ('s' if ny == 24 else '^')
        ax.scatter(x + jitter, np.maximum(sub[value_col].to_numpy(float), floor), s=14, alpha=0.55,
                   color=ny_colors[ny], marker=marker, linewidths=0, label=fr'$N_y={ny}$ samples')
        m = mean_spread[mean_spread['Ny'] == ny]
        ax.plot(m['alpha_topological_region'], np.maximum(m[value_col], floor), color=ny_colors[ny], lw=1.45,
                marker=marker, ms=3.2, label=fr'$N_y={ny}$ mean')
    ax.axvline(2.0, color='0.35', linestyle=':', lw=1.1)
    ax.set_yscale('log')
    ax.set_xlabel(r'$\alpha_1$')
    ax.set_ylabel(ylabel)
    ax.set_title(f'Perfect correction: {title}')
    ax.set_xticks(spread_alphas[::2])
axes[0].text(0.02, 0.03, 'values below display floor clipped', transform=axes[0].transAxes, fontsize=6, color='0.35')
handles, labels = axes[1].get_legend_handles_labels()
mean_handles = [(h, l) for h, l in zip(handles, labels) if 'mean' in l]
axes[1].legend([h for h, _ in mean_handles], [l for _, l in mean_handles], loc='upper right', frameon=False)
savefig(fig, 'fine21_perfect_correction_final_spread_crossover_with_means')
plt.show()


## Cycle Dynamics At Representative $lpha_1$ Values


In [ ]:
selected_alphas = [1.0, 1.5, 1.8, 2.0, 3.0]
fig, axes = plt.subplots(1, len(selected_alphas), figsize=(10.125, 2.45), sharey=True, constrained_layout=True)
for ax, alpha_value in zip(axes, selected_alphas):
    for protocol in protocols:
        for ny in ny_values:
            sub = dynamics_df[(dynamics_df['protocol'] == protocol) & (dynamics_df['Ny'] == ny) & np.isclose(dynamics_df['alpha_topological_region'], alpha_value)]
            style = protocol_styles[protocol]
            ax.plot(sub['cycle'], sub['entropy_mean'], color=ny_colors[ny], linestyle=style['linestyle'], lw=1.25,
                    label=fr'$N_y={ny}$, {style["label"]}')
            if protocol == 'perfect_correction':
                ax.fill_between(sub['cycle'], sub['entropy_mean'] - sub['entropy_std'], sub['entropy_mean'] + sub['entropy_std'],
                                color=ny_colors[ny], alpha=0.12, lw=0)
    ax.set_title(fr'$\alpha_1={alpha_value:g}$')
    ax.set_xlabel('cycle')
axes[0].set_ylabel(r'$S_{\mathrm{global}}$')
axes[-1].legend(ncol=1, loc='center left', bbox_to_anchor=(1.02, 0.5), frameon=False)
savefig(fig, 'fine21_entropy_dynamics_representative_alphas')
plt.show()


## Charge-Variance Dynamics At Representative $lpha_1$ Values


In [ ]:
selected_alphas = [1.0, 1.5, 1.8, 2.0, 3.0]
fig, axes = plt.subplots(1, len(selected_alphas), figsize=(10.125, 2.45), sharey=True, constrained_layout=True)
for ax, alpha_value in zip(axes, selected_alphas):
    for protocol in protocols:
        for ny in ny_values:
            sub = dynamics_df[(dynamics_df['protocol'] == protocol) & (dynamics_df['Ny'] == ny) & np.isclose(dynamics_df['alpha_topological_region'], alpha_value)]
            style = protocol_styles[protocol]
            ax.plot(sub['cycle'], sub['charge_variance_mean'], color=ny_colors[ny], linestyle=style['linestyle'], lw=1.25,
                    label=fr'$N_y={ny}$, {style["label"]}')
            if protocol == 'perfect_correction':
                ax.fill_between(sub['cycle'], sub['charge_variance_mean'] - sub['charge_variance_std'], sub['charge_variance_mean'] + sub['charge_variance_std'],
                                color=ny_colors[ny], alpha=0.12, lw=0)
    ax.set_title(fr'$\alpha_1={alpha_value:g}$')
    ax.set_xlabel('cycle')
axes[0].set_ylabel(r'$\delta Q^2$')
axes[-1].legend(ncol=1, loc='center left', bbox_to_anchor=(1.02, 0.5), frameon=False)
savefig(fig, 'fine21_charge_variance_dynamics_representative_alphas')
plt.show()


## Log-Scale Late-Time Dynamics

These panels use logarithmic vertical scales to show the late-time collapse that is visually compressed in the linear dynamics figures.

In [ ]:

selected_alphas = [1.0, 1.5, 1.8, 2.0, 3.0]
fig, axes = plt.subplots(1, len(selected_alphas), figsize=(10.125, 2.45), sharey=True, constrained_layout=True)
for ax, alpha_value in zip(axes, selected_alphas):
    for protocol in protocols:
        for ny in ny_values:
            sub = dynamics_df[(dynamics_df['protocol'] == protocol) & (dynamics_df['Ny'] == ny) & np.isclose(dynamics_df['alpha_topological_region'], alpha_value)]
            style = protocol_styles[protocol]
            y = np.maximum(sub['entropy_mean'].to_numpy(float), 1e-8)
            ax.plot(sub['cycle'], y, color=ny_colors[ny], linestyle=style['linestyle'], lw=1.25,
                    label=fr'$N_y={ny}$, {style["label"]}')
            if protocol == 'perfect_correction':
                low = np.maximum((sub['entropy_mean'] - sub['entropy_std']).to_numpy(float), 1e-8)
                high = np.maximum((sub['entropy_mean'] + sub['entropy_std']).to_numpy(float), 1e-8)
                ax.fill_between(sub['cycle'], low, high, color=ny_colors[ny], alpha=0.12, lw=0)
    ax.set_yscale('log')
    ax.set_title(fr'$\alpha_1={alpha_value:g}$')
    ax.set_xlabel('cycle')
axes[0].set_ylabel(r'$S_{\mathrm{global}}$')
axes[-1].legend(ncol=1, loc='center left', bbox_to_anchor=(1.02, 0.5), frameon=False)
savefig(fig, 'fine21_entropy_dynamics_representative_alphas_log')
plt.show()


In [ ]:

selected_alphas = [1.0, 1.5, 1.8, 2.0, 3.0]
fig, axes = plt.subplots(1, len(selected_alphas), figsize=(10.125, 2.45), sharey=True, constrained_layout=True)
for ax, alpha_value in zip(axes, selected_alphas):
    for protocol in protocols:
        for ny in ny_values:
            sub = dynamics_df[(dynamics_df['protocol'] == protocol) & (dynamics_df['Ny'] == ny) & np.isclose(dynamics_df['alpha_topological_region'], alpha_value)]
            style = protocol_styles[protocol]
            y = np.maximum(sub['charge_variance_mean'].to_numpy(float), 1e-12)
            ax.plot(sub['cycle'], y, color=ny_colors[ny], linestyle=style['linestyle'], lw=1.25,
                    label=fr'$N_y={ny}$, {style["label"]}')
            if protocol == 'perfect_correction':
                low = np.maximum((sub['charge_variance_mean'] - sub['charge_variance_std']).to_numpy(float), 1e-12)
                high = np.maximum((sub['charge_variance_mean'] + sub['charge_variance_std']).to_numpy(float), 1e-12)
                ax.fill_between(sub['cycle'], low, high, color=ny_colors[ny], alpha=0.12, lw=0)
    ax.set_yscale('log')
    ax.set_title(fr'$\alpha_1={alpha_value:g}$')
    ax.set_xlabel('cycle')
axes[0].set_ylabel(r'$\delta Q^2$')
axes[-1].legend(ncol=1, loc='center left', bbox_to_anchor=(1.02, 0.5), frameon=False)
savefig(fig, 'fine21_charge_variance_dynamics_representative_alphas_log')
plt.show()


## Text Summary Tables


In [ ]:
# Compact tables for interpretation in the writeup.
interesting_alphas = [1.0, 1.5, 1.8, 1.9, 2.0, 2.1, 3.0]
compact = final_summary[final_summary['alpha_topological_region'].round(10).isin(interesting_alphas)].copy()
compact = compact[['protocol', 'Ny', 'alpha_topological_region', 'samples', 'entropy_mean', 'entropy_std',
                   'charge_variance_mean', 'charge_variance_std', 'sharpening_fraction']]
compact.to_csv(TABLE_ROOT / 'interpretation_key_alpha_summary.csv', index=False)
display(compact)

# Estimate where each curve first has mean charge variance below the raw threshold, and first has sharpening_fraction >= 0.5.
rows = []
for (protocol, ny), group in final_summary.groupby(['protocol', 'Ny']):
    group = group.sort_values('alpha_topological_region')
    below_mean = group[group['charge_variance_mean'] < SHARPENING_THRESHOLD]
    frac_half = group[group['sharpening_fraction'] >= 0.5]
    rows.append({
        'protocol': protocol,
        'Ny': int(ny),
        'first_alpha_mean_deltaQ2_below_1e-2': float(below_mean['alpha_topological_region'].iloc[0]) if len(below_mean) else np.nan,
        'first_alpha_sharpening_fraction_ge_0p5': float(frac_half['alpha_topological_region'].iloc[0]) if len(frac_half) else np.nan,
        'first_alpha_sharpening_fraction_eq_1': float(group[group['sharpening_fraction'] >= 1.0]['alpha_topological_region'].iloc[0]) if len(group[group['sharpening_fraction'] >= 1.0]) else np.nan,
    })
transition_table = pd.DataFrame(rows)
transition_table.to_csv(TABLE_ROOT / 'threshold_crossing_summary.csv', index=False)
display(transition_table)
